# Hub-and-Spoke Agents with the Claude Agent SDK

**Pattern:** one *hub* (orchestrator) agent receives the request, breaks it down, and delegates to specialized *spoke* subagents. Spokes never talk to each other; everything flows through the hub, which merges their results into the final answer.

```
              +-------------+
              |     HUB     |  plans, delegates, synthesizes
              +------+------+
        +------------+------------+
        v            v            v
  researcher      analyst       writer
```

In the SDK, spokes are `AgentDefinition`s passed via `agents=`, and the hub invokes them through the built-in subagent tool (`Task`, called `Agent` in newer versions).

In [ ]:
!pip install -q claude-agent-sdk

In [ ]:
import os, getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

## 1. Define the spokes

Each spoke gets a focused prompt and only the tools it needs. The `description` is what the hub reads to decide *when* to delegate, so make it specific.

In [ ]:
from claude_agent_sdk import AgentDefinition

spokes = {
    'researcher': AgentDefinition(
        description='Finds facts on a topic via web search. Use for any question needing current or factual information.',
        prompt=(
            'You are a research specialist. Search the web, verify claims across sources, '
            'and return concise bullet-point findings with source URLs. No opinions, no fluff.'
        ),
        tools=['WebSearch', 'WebFetch'],
        model='sonnet',
    ),
    'analyst': AgentDefinition(
        description='Compares options, weighs trade-offs, and draws conclusions from facts already gathered.',
        prompt=(
            'You are an analyst. Given findings, compare the options on clear criteria, '
            'call out trade-offs and risks, and give a justified recommendation.'
        ),
        tools=['Read'],
        model='sonnet',
    ),
    'writer': AgentDefinition(
        description='Turns findings and analysis into a polished, readable final deliverable.',
        prompt=(
            'You are a technical writer. Produce a clear, well-structured summary '
            'from the material you are given. Do not invent facts.'
        ),
        tools=['Read'],
        model='haiku',
    ),
}

## 2. Define the hub

The hub's system prompt tells it to *delegate rather than do the work itself*. Allowing the subagent tool is what lets it call the spokes.

In [ ]:
from claude_agent_sdk import ClaudeAgentOptions

HUB_PROMPT = '''You are the hub of a multi-agent team. You do NOT research, analyze, or write yourself.
Your job:
1. Break the user's request into steps.
2. Delegate each step to the right specialist: researcher, analyst, or writer.
3. Pass the output of earlier specialists to later ones (they cannot see each other).
4. Return the writer's final result to the user.'''

options = ClaudeAgentOptions(
    system_prompt=HUB_PROMPT,
    agents=spokes,
    # Subagent tool name differs across SDK versions. The spokes' tools must be
    # pre-approved here too, or unattended runs deny them (e.g. web search).
    allowed_tools=['Task', 'Agent', 'WebSearch', 'WebFetch', 'Read'],
    model='sonnet',
    max_turns=25,
)

## 3. Run it

This helper streams the conversation and prints each delegation so you can see the hub routing work to spokes.

In [ ]:
from claude_agent_sdk import query, AssistantMessage, TextBlock, ToolUseBlock, ResultMessage

async def run(prompt: str):
    async for msg in query(prompt=prompt, options=options):
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, ToolUseBlock) and block.name in ('Task', 'Agent'):
                    spoke = block.input.get('subagent_type')
                    print(f"[hub -> {spoke}] {block.input.get('description', '')}")
                elif isinstance(block, TextBlock) and getattr(msg, 'parent_tool_use_id', None) is None:
                    print(block.text)
        elif isinstance(msg, ResultMessage):
            cost = getattr(msg, 'total_cost_usd', None)
            print(f"\n--- done in {msg.num_turns} turns" + (f", ${cost:.4f}" if cost else '') + ' ---')

# Notebooks support top-level await
await run('Compare Postgres and CockroachDB for a multi-region SaaS app and give me a one-page recommendation.')

## What to look for

- Output lines like `[hub -> researcher]`, then `[hub -> analyst]`, then `[hub -> writer]` show the hub routing work.
- Text from spokes stays inside their own context. Only the hub's messages are printed (`parent_tool_use_id is None`).
- To extend: add another `AgentDefinition` (e.g. `critic`) and mention it in `HUB_PROMPT`.